# Phase 4 · Context windows of two and three tokens

James Webb · 2026-11

I predict the wider window will pick up more neighbors because it can reach one extra token on either side. Both runs use the original 22 documents and the example tokenizer.

Adapted from [Denise Case’s example](https://github.com/denisecase/nlp-03-text-exploration) with ChatGPT assistance.

In [1]:
from collections import Counter


def tokenize(text: str) -> list[str]:
    """Tokenize the input text into a list of cleaned tokens."""
    tokens = text.lower().split()
    return [t.strip(".,:;!?()[]\"'") for t in tokens if len(t) > 2]


def nearby_words(documents, target, radius):
    """Count neighbors of the target within each document."""
    neighbors = Counter()
    for document in documents:
        tokens = tokenize(document["text"])
        for index, token in enumerate(tokens):
            if token == target:
                start = max(0, index - radius)
                end = min(len(tokens), index + radius + 1)
                neighbors.update(word for word in tokens[start:end] if word != target)
    return neighbors


corpus: list[dict[str, str]] = [
    # Dogs
    {"category": "dog", "text": "A dog barks loudly."},
    {"category": "dog", "text": "The puppy runs in the yard."},
    {"category": "dog", "text": "A canine wears a leash."},
    {"category": "dog", "text": "The kennel holds the dog."},
    {"category": "dog", "text": "The dog ran across the yard."},
    {"category": "dog", "text": "The puppy ran across the yard."},
    # Cats
    {"category": "cat", "text": "A cat sleeps quietly."},
    {"category": "cat", "text": "The kitten plays with yarn."},
    {"category": "cat", "text": "A feline purrs softly."},
    {"category": "cat", "text": "The cat has whiskers."},
    {"category": "cat", "text": "The cat slept near the window."},
    {"category": "cat", "text": "The kitten slept near the window."},
    # Cars
    {"category": "car", "text": "A car drives on the road."},
    {"category": "car", "text": "The sedan parks in the garage."},
    {"category": "car", "text": "A vehicle has four wheels."},
    {"category": "car", "text": "The car moves down the highway."},
    {"category": "car", "text": "The car stopped near the garage."},
    {"category": "car", "text": "The sedan stopped near the garage."},
    # Trucks
    {"category": "truck", "text": "A truck carries cargo."},
    {"category": "truck", "text": "The pickup pulls a trailer."},
    {"category": "truck", "text": "The engine powers the truck."},
    {"category": "truck", "text": "The truck hauls heavy loads."},
]

In [2]:
for target in ("dog", "cat", "car", "truck"):
    two = nearby_words(corpus, target, 2)
    three = nearby_words(corpus, target, 3)
    added = sorted(set(three) - set(two))
    print(
        f"{target}: {two.total()} → {three.total()} neighbor occurrences "
        f"| newly visible words: {', '.join(added) or 'none'}"
    )
    assert all(three[word] >= count for word, count in two.items())

dog: 7 → 9 neighbor occurrences | newly visible words: kennel
cat: 8 → 9 neighbor occurrences | newly visible words: none
car: 8 → 11 neighbor occurrences | newly visible words: road
truck: 7 → 9 neighbor occurrences | newly visible words: engine, loads


## Interpretation

The wider window increased the number of neighbor occurrences for each target. It can add content words and common words, so a larger count alone does not establish a stronger relationship.